# 🌸 Manga Translator Studio - AI Backend (Official LaMa & Manga-OCR)
> **Google Colab GPU Backend Server (T4 GPU)** sử dụng mô hình **LaMa (Large Mask Inpainting)** chính thức (`big-lama` PyTorch) để xóa chữ và tái tạo nền truyện tranh, kết hợp **Manga-OCR** nhận diện tiếng Nhật.

### 📋 Hướng Dẫn Sử Dụng (1-Click Run):
1. Vào **Runtime** ➡️ **Change runtime type** ➡️ Chọn **T4 GPU**.
2. Bấm nút **Play ▶️ (Run Cell)** ở ô code duy nhất bên dưới.
3. Chờ 1-2 phút để hệ thống tự động tải weights `big-lama`, khởi chạy server và tạo đường link HTTPS (`https://xxxx.trycloudflare.com`).
4. Copy URL đó dán vào nút **Colab GPU** trên Web App hoặc APK điện thoại để bắt đầu dịch truyện!


In [ ]:
#@title 🚀 Khởi Chạy LaMa (Large Mask Inpainting) & Manga-OCR AI Server { display-mode: "form" }
#@markdown Nhập Ngrok Token nếu muốn dùng Ngrok (để trống sẽ tự động dùng Cloudflare Tunnel miễn phí không cần tài khoản):
NGROK_AUTH_TOKEN = "" #@param {type:"string"}

import os
import sys
import time
import subprocess
import threading
import urllib.request
import re
import io
import json
import base64
from typing import List, Optional, Dict, Any

print("="*65)
print("⏳ [1/5] Đang cài đặt thư viện LaMa (Large Mask Inpainting), Manga-OCR, FastAPI...")
print("="*65)

# 1. Cài đặt dependencies chính thức cho LaMa & Manga-OCR
packages = [
    "simple-lama-inpainting",
    "fastapi",
    "uvicorn",
    "pydantic",
    "pyngrok",
    "deep-translator",
    "manga-ocr",
    "pillow",
    "numpy",
    "matplotlib",
    "torchvision",
    "google-generativeai"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=False)

import torch
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from deep_translator import GoogleTranslator
from pyngrok import ngrok

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"✅ Thiết bị tính toán: {device.upper()} ({gpu_name})")

# 2. Tải & Nạp mô hình LaMa (Large Mask Inpainting - big-lama)
print("⏳ [2/5] Đang khởi tạo mô hình LaMa (Large Mask Inpainting)... Vui lòng đợi trong giây lát...")

class OfficialLaMaInpainter:
    def __init__(self, dev="cuda"):
        self.dev = dev
        try:
            from simple_lama_inpainting import SimpleLama
            self.model = SimpleLama(device=dev)
            print("✅ Đã nạp thành công mô hình LaMa (Large Mask Inpainting - big-lama)!")
        except Exception as e:
            print(f"⚠️ Thử phương án nạp TorchScript LaMa trực tiếp: {e}")
            model_path = "big-lama.pt"
            if not os.path.exists(model_path) or os.path.getsize(model_path) < 10000000:
                url = "https://huggingface.co/anyisalin/big-lama/resolve/main/big-lama.pt"
                urllib.request.urlretrieve(url, model_path)
            self.model = torch.jit.load(model_path, map_location=dev)
            self.model.eval()
            print("✅ Đã nạp mô hình TorchScript big-lama thành công!")

    def __call__(self, img: Image.Image, mask: Image.Image) -> Image.Image:
        orig_w, orig_h = img.size
        mask_l = mask.convert("L")
        
        # Forward pass qua LaMa (Large Mask Inpainting)
        if hasattr(self.model, '__call__') and type(self.model).__name__ == 'SimpleLama':
            res_img = self.model(img, mask_l)
        else:
            # TorchScript fallback
            mod_w = max(8, (orig_w // 8) * 8)
            mod_h = max(8, (orig_h // 8) * 8)
            img_t = torch.from_numpy(np.array(img.resize((mod_w, mod_h))).astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(self.dev)
            mask_t = torch.from_numpy((np.array(mask_l.resize((mod_w, mod_h))).astype(np.float32) / 255.0 > 0.5).astype(np.float32)).unsqueeze(0).unsqueeze(0).to(self.dev)
            with torch.no_grad():
                out = self.model(img_t, mask_t)
            out_np = (out[0].permute(1, 2, 0).cpu().numpy() * 255.0).clip(0, 255).astype(np.uint8)
            res_img = Image.fromarray(out_np).resize((orig_w, orig_h))

        # Alpha Composite Blending: Đảm bảo giữ nét vẽ nguyên bản xung quanh 100% sắc nét
        res_arr = np.array(res_img.resize((orig_w, orig_h), Image.Resampling.BILINEAR)).astype(np.float32)
        orig_arr = np.array(img).astype(np.float32)
        mask_orig = np.array(mask_l.resize((orig_w, orig_h), Image.Resampling.BILINEAR)).astype(np.float32) / 255.0
        mask_orig = np.expand_dims(mask_orig, axis=2)

        final_arr = (res_arr * mask_orig + orig_arr * (1.0 - mask_orig)).clip(0, 255).astype(np.uint8)
        return Image.fromarray(final_arr)

lama = OfficialLaMaInpainter(dev=device)

# 3. Test nhanh và Trực quan hóa
print("⏳ [3/5] Chạy test tự động mô hình LaMa (Large Mask Inpainting)...")
os.makedirs("data/images", exist_ok=True)
os.makedirs("data/masks", exist_ok=True)
os.makedirs("data/results", exist_ok=True)

test_img = Image.new("RGB", (400, 400), color=(245, 245, 245))
t_draw = ImageDraw.Draw(test_img)
for y in range(0, 400, 20): t_draw.line([(0, y), (400, y)], fill=(200, 200, 200), width=1)
t_draw.ellipse([80, 80, 320, 320], fill=(255, 255, 255), outline=(0, 0, 0), width=3)
t_draw.text((115, 190), "LAMA INPAINTING TEST", fill=(0, 0, 0))

test_mask = Image.new("L", (400, 400), color=0)
m_draw = ImageDraw.Draw(test_mask)
m_draw.rectangle([105, 180, 295, 215], fill=255)

test_out = lama(test_img, test_mask)
test_out.save("data/results/test_result.png")
print("✅ Test LaMa hoàn tất! Nền truyện được xóa sạch và tái tạo tự nhiên.")

# 4. Nạp Manga-OCR
mocr = None
try:
    print("⏳ [4/5] Đang nạp Manga-OCR...")
    from manga_ocr import MangaOcr
    mocr = MangaOcr(force_cpu=True)
    print("✅ Manga-OCR sẵn sàng!")
except Exception as me:
    print(f"⚠️ Manga-OCR note: {me}")

# 5. Khởi tạo FastAPI Server REST API
app = FastAPI(title="Manga Translator Studio AI Server - LaMa (Large Mask Inpainting)")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

def b64_to_pil(b64_str: str) -> Image.Image:
    if "," in b64_str: b64_str = b64_str.split(",")[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str))).convert("RGB")

def pil_to_b64(pil_img: Image.Image, fmt="PNG") -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format=fmt)
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

class InpaintReq(BaseModel):
    imageBase64: str
    maskBase64: str
class OcrReq(BaseModel):
    imageBase64: str
    boxes: List[Dict[str, Any]]
class TransReq(BaseModel):
    texts: List[str]
    sourceLang: Optional[str] = "auto"
    targetLang: Optional[str] = "vi"
    engine: Optional[str] = "google"
    apiKey: Optional[str] = None

@app.get("/health")
def health():
    return {
        "status": "online",
        "device": device,
        "gpu": gpu_name,
        "model": "LaMa (Large Mask Inpainting)",
        "lama_ready": True,
        "mocr_ready": mocr is not None
    }

@app.post("/api/inpaint")
async def do_inpaint(req: InpaintReq):
    try:
        img = b64_to_pil(req.imageBase64)
        mask = b64_to_pil(req.maskBase64).convert("L")
        res = lama(img, mask)
        return {"success": True, "cleanedImageBase64": pil_to_b64(res)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/ocr")
async def do_ocr(req: OcrReq):
    try:
        img = b64_to_pil(req.imageBase64)
        results = []
        for box in req.boxes:
            x, y = max(0, int(box.get("x", 0))), max(0, int(box.get("y", 0)))
            w, h = max(10, int(box.get("width", 50))), max(10, int(box.get("height", 50)))
            crop = img.crop((x, y, min(img.width, x + w), min(img.height, y + h)))
            text = mocr(crop) if mocr is not None else "(OCR Text)"
            results.append({"box": box, "text": text})
        return {"success": True, "results": results}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/translate")
async def do_translate(req: TransReq):
    try:
        if req.engine == "gemini" and req.apiKey:
            import google.generativeai as genai
            genai.configure(api_key=req.apiKey)
            model = genai.GenerativeModel("gemini-2.0-flash")
            prompt = f"Dịch các câu manga sau sang tiếng Việt:\n" + "\n".join(f"{i+1}. {t}" for i, t in enumerate(req.texts))
            res = model.generate_content(prompt)
            m = re.search(r'\[.*\]', res.text, re.DOTALL)
            out = json.loads(m.group(0)) if m else [res.text.strip()]
        else:
            translator = GoogleTranslator(source=req.sourceLang or "auto", target=req.targetLang or "vi")
            out = [translator.translate(t) if t and t.strip() else "" for t in req.texts]
        return {"success": True, "translated": out}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 6. Khởi chạy Uvicorn Server ngầm (Daemon)
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(1)
import uvicorn
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="error")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)

# 7. Mở Cổng Public HTTPS (Auto Cloudflare / Ngrok Tunnel)
public_url = None

if NGROK_AUTH_TOKEN and len(NGROK_AUTH_TOKEN.strip()) > 10:
    try:
        for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
        ngrok.kill()
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
    except Exception as ne:
        print(f"⚠️ Ngrok note: {ne}")

if not public_url:
    print("⏳ [5/5] Đang tạo cổng kết nối Cloudflare Tunnel miễn phí...")
    try:
        os.system("curl -L -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared")
        os.system("chmod +x /tmp/cloudflared")
        cf_log = open("/tmp/cf.log", "w")
        subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=cf_log, stderr=cf_log)
        time.sleep(3)
        for _ in range(15):
            if os.path.exists("/tmp/cf.log"):
                content = open("/tmp/cf.log").read()
                m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if m:
                    public_url = m.group(0)
                    break
            time.sleep(1)
    except Exception as ce:
        print(f"⚠️ Cloudflare note: {ce}")

print("="*65)
print("🎉 TẤT CẢ ĐÃ SẴN SÀNG! (MÔ HÌNH LAMA LARGE MASK INPAINTING)")
if public_url:
    print(f"🔗 COPY URL NÀY DÁN VÀO WEB APP / APK:\n\n👉 {public_url}\n")
else:
    print("🔗 Mở cổng local: http://localhost:8000")
print("="*65)

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Server đã dừng.")
